In [63]:
pip install pandas streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [64]:
import ast
import pandas as pd

In [65]:
movies=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_movies.csv')
credits=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_credits.csv')


In [66]:
movies = movies.merge(credits, left_on='original_title', right_on='title')
movies.drop(columns='original_title',inplace=True)


In [67]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')

In [68]:
movies = movies.rename(columns={'title_x': 'title'})

In [69]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [70]:
def convert_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:
        L.append(i['name'])
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [71]:
movies.isnull().sum()
movies.dropna(inplace=True)

In [72]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4544 entries, 0 to 4546
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4544 non-null   int64 
 1   title     4544 non-null   object
 2   overview  4544 non-null   object
 3   genres    4544 non-null   object
 4   keywords  4544 non-null   object
 5   cast      4544 non-null   object
 6   crew      4544 non-null   object
dtypes: int64(1), object(6)
memory usage: 284.0+ KB


In [73]:
print(movies['genres'].iloc[0])


[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [ ]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)


In [74]:
def fetch_director(text):
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            return i['name']
    return ""

movies['crew'] = movies['crew'].apply(fetch_director)

In [75]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['crew'] = movies['crew'].apply(lambda x: [x])

In [76]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [77]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [78]:
import requests

API_KEY = "eb907f1e"

def fetch_poster(movie_title):
    url = f"http://www.omdbapi.com/?t={movie_title}&apikey={API_KEY}"
    
    try:
        data = requests.get(url, timeout=5).json()
        
        if data.get("Response") == "True":
            poster = data.get("Poster")
            
            # Handle cases where poster is "N/A"
            if poster and poster != "N/A":
                return poster
        
        return "https://via.placeholder.com/300x450?text=No+Image"
    
    except:
        return "https://via.placeholder.com/300x450?text=Error"

In [79]:
print(fetch_poster("vikings"))

https://m.media-amazon.com/images/M/MV5BOTFmZmExYTEtYmE0Mi00MzRmLWE4ZDYtOThiNzNlOTIyODljXkEyXkFqcGc@._V1_SX300.jpg
